In [16]:
!pip install pymupdf pandas openpyxl
import fitz
import pandas as pd
import re
import math
import os
import string
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter
from openpyxl.cell.rich_text import TextBlock, CellRichText
from openpyxl.cell.text import InlineFont
from openpyxl.worksheet.datavalidation import DataValidation

# --- 檔案名稱設定區 ---
EXCEL_FILE = "BOM.xlsx"
PDF_FILE = "SCH.pdf"
OUTPUT_EXCEL = "SCH & BOM 比對結果.xlsx"
COLUMN_NAME = "Location"

def apply_mixed_font(cell, text, size=11, bold=False, color="4F4F4F"):
    """莫蘭迪風格：微軟正黑體 + Segoe UI 混排"""
    font_zh = InlineFont(rFont='微軟正黑體', sz=size, b=bold, color=color)
    font_en = InlineFont(rFont='Segoe UI', sz=size, b=bold, color=color)

    rt = CellRichText()
    parts = re.split(r'([\u4e00-\u9fff]+)', text)

    for part in parts:
        if not part: continue
        if re.search(r'[\u4e00-\u9fff]', part):
            rt.append(TextBlock(font_zh, part))
        else:
            rt.append(TextBlock(font_en, part))
    cell.value = rt

def get_sort_key(refdes):
    ref_str = str(refdes).strip()
    if not ref_str or ref_str.lower() == 'nan': return (9999, 0, ref_str)
    match = re.match(r"([A-Za-z]+)([0-9]+)?", ref_str)
    SORT_ORDER = ["PCB","inlet", "F", "CX", "CK","CR", "L", "LF", "T", "RA", "RB", "RC", "RD", "RE", "R", "RX", "M", "TH", "C", "CY", "B", "BD", "ACD", "ZD", "D", "Q", "PC", "U"]
    order_lower = [s.lower() for s in SORT_ORDER]
    if not match: return (-2, 0, ref_str)
    prefix, num_str = match.groups()
    prefix_lower = prefix.lower()
    num = int(num_str) if num_str else 0
    category_weight = order_lower.index(prefix_lower) if prefix_lower in order_lower else 999
    return (category_weight, num)

def get_downward_score(ref_rect, other_rect):
    """
    計算正下方的偏好得分：
    1. other_rect 必須在 ref_rect 的下方 (oth_y0 >= ref_y0 - 2)
    2. X 軸重疊度高或中心距離近者優先
    3. 垂直距離越近越好
    """
    ref_cx = (ref_rect.x0 + ref_rect.x1) / 2
    oth_cx = (other_rect.x0 + other_rect.x1) / 2

    # Y 軸必須在下方
    if other_rect.y0 < ref_rect.y0 - 2:
        return float('inf')

    dy = other_rect.y0 - ref_rect.y1
    if dy < -5 or dy > 120:  # 垂直距離過遠或在上方
        return float('inf')

    dx = abs(oth_cx - ref_cx)

    if dx > 45:
        return float('inf')

    return dy + (dx * 3.0)

def get_right_score(ref_rect, other_rect):
    """
    備用：右側文字得分（適用於 IC 如 U52 右側的 RT7211）
    """
    ref_cy = (ref_rect.y0 + ref_rect.y1) / 2
    oth_cy = (other_rect.y0 + other_rect.y1) / 2

    # X 軸必須在右側
    if other_rect.x0 < ref_rect.x0 - 2:
        return float('inf')

    dx = other_rect.x0 - ref_rect.x1
    dy = abs(oth_cy - ref_cy)

    if dx < -2 or dx > 80 or dy > 10:
        return float('inf')

    return dx + (dy * 2.0)

def start_process():
    print(f"🚀 啟動完整校正程序：分析 {PDF_FILE} ...")
    try:
        # 1. 讀取 BOM 資料 (動態尋找標頭行)
        df_bom_raw = pd.read_excel(EXCEL_FILE, header=None)

        header_row_idx = None
        for idx, row in df_bom_raw.iterrows():
            if row.astype(str).str.strip().eq(COLUMN_NAME).any():
                header_row_idx = idx
                break

        if header_row_idx is None:
            raise ValueError(f"❌ 在 BOM 中完全找不到欄位名稱: '{COLUMN_NAME}'，請檢查 Excel！")

        df_bom = pd.read_excel(EXCEL_FILE, skiprows=header_row_idx)

        new_headers = [str(df_bom.columns[i]) for i in range(min(4, len(df_bom.columns)))]
        while len(new_headers) < 4:
            new_headers.append(f"BOM_Col_{len(new_headers)+1}")

        df_data = df_bom[df_bom[COLUMN_NAME].notna()].copy()
        designators = df_data[COLUMN_NAME].astype(str).str.strip()

        maps = []
        for i in range(4):
            col_name = df_bom.columns[i]
            maps.append(dict(zip(designators, df_data[col_name])))

        bom_refdes = [ref for ref in designators.unique() if ref.lower() not in [COLUMN_NAME.lower(), 'nan', '']]

        results_map = {ref: {"元件值": "未找到", "元件封裝": "未找到", "是否在PDF出現": "❌ 否",
                             "orig": [m.get(ref, "") for m in maps]} for ref in bom_refdes}

        # 2. PDF 處理
        doc = fitz.open(PDF_FILE)
        page_images = []
        PKG_PATTERNS = r"0201|0402|0603|0805|1206|SOT|QFN|SOD|DO-|RES-|CAP-|TQFP|BGA|LED|10x16"

        for i, page in enumerate(doc):
            # 擷取頁面上所有的單詞或文字區塊
            text_instances = page.get_text("words")  # (x0, y0, x1, y1, word, block_no, line_no, word_no)

            all_spans = []
            blocks = page.get_text("dict")["blocks"]
            for b in blocks:
                if "lines" in b:
                    for l in b["lines"]:
                        for s in l["spans"]:
                            txt = s["text"].strip()
                            if txt:
                                all_spans.append({"text": txt, "rect": fitz.Rect(s["bbox"])})

            for ref in bom_refdes:
                # 定義嚴格匹配 pattern (包含獨立單詞、帶加號如 +U52、或帶後綴如 U52-A)
                pattern = r"(?:^|[^\w])\+?" + re.escape(ref) + r"(?:-[A-Z0-9]+)?(?:$|[^\w])"

                matched_item = None

                for item in all_spans:
                    if re.search(pattern, item["text"]):
                        matched_item = item
                        break

                # 若無，搜尋整個頁面文字
                if not matched_item:
                    rects = page.search_for(ref)
                    if rects:
                        matched_item = {"text": ref, "rect": rects[0]}

                if matched_item:
                    results_map[ref]["是否在PDF出現"] = "✅ 是"
                    ref_rect = matched_item["rect"]

                    # 搜尋正下方的文字 (Down candidates)
                    down_cands = []
                    # 搜尋右側的文字 (Right candidates)
                    right_cands = []

                    for o in all_spans:
                        o_txt = o["text"].strip()
                        # 跳過自身與包含元件代號的文字
                        if re.search(pattern, o_txt):
                            continue

                        d_score = get_downward_score(ref_rect, o["rect"])
                        if d_score != float('inf'):
                            down_cands.append((d_score, o_txt, o["rect"]))

                        r_score = get_right_score(ref_rect, o["rect"])
                        if r_score != float('inf'):
                            right_cands.append((r_score, o_txt, o["rect"]))

                    down_cands.sort(key=lambda x: x[0])
                    right_cands.sort(key=lambda x: x[0])

                    # 賦值邏輯：優先採用正下方的文字 (例如 R38 往下找到 22R F 與 0603)
                    if len(down_cands) >= 2:
                        val1, val2 = down_cands[0][1], down_cands[1][1]
                        is_first_pkg = re.search(PKG_PATTERNS, val1, re.I)
                        if is_first_pkg:
                            results_map[ref]["元件值"], results_map[ref]["元件封裝"] = val2, val1
                        else:
                            results_map[ref]["元件值"], results_map[ref]["元件封裝"] = val1, val2
                    elif len(down_cands) == 1:
                        results_map[ref]["元件值"] = down_cands[0][1]
                    elif len(right_cands) >= 1:
                        # 正下方沒有文字時（如 IC U52 型號寫在右邊）
                        results_map[ref]["元件值"] = right_cands[0][1]

                    # 標註黃色螢光
                    annot = page.add_highlight_annot(ref_rect)
                    if annot:
                        annot.set_colors(stroke=(1, 1, 0)) # 亮黃色
                        annot.update()

            # --- 渲染高解析度圖片 ---
            zoom_matrix = fitz.Matrix(4, 4)
            pix = page.get_pixmap(matrix=zoom_matrix)
            img_path = f"temp_sch_page_{i}.png"
            pix.save(img_path)
            page_images.append(img_path)

        # 3. 整理資料
        final_list = [{"元件代號": k, "元件值": v["元件值"], "元件封裝": v["元件封裝"], "是否在PDF出現": v["是否在PDF出現"],
                       new_headers[0]: v["orig"][0], new_headers[1]: v["orig"][1],
                       new_headers[2]: v["orig"][2], new_headers[3]: v["orig"][3]} for k, v in results_map.items()]
        final_df = pd.DataFrame(final_list)
        final_df['sort_key'] = final_df['元件代號'].apply(get_sort_key)
        final_df = final_df.sort_values(by='sort_key').drop(columns=['sort_key'])

        # 4. Excel 輸出與釘選
        with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
            final_df.to_excel(writer, index=False, sheet_name='CheckList', startrow=2)
            ws = writer.sheets['CheckList']
            max_row = ws.max_row
            ws.freeze_panes = 'A4'

            # 莫蘭迪配色
            M_BLUE_DARK, M_BLUE_LIGHT, M_PINK, M_GREEN, M_BEIGE, M_TEXT, M_BORDER = "8294A5", "E1E7ED", "E8D7D7", "CFD8D0", "F2F1EF", "4F4F4F", "D1D1D1"

            ws.merge_cells('A1:H2')
            header_text = ("【SCH & BOM 比對報告】 A~C 欄：電路圖自動擷取資訊 │ E~H 欄：原始 BOM 對應資訊。\n"
                           " ⚠️ D欄標註「❌ 否」代表該元件於 PDF 中未偵測到，請核對代號或確認是否漏畫。")
            apply_mixed_font(ws['A1'], header_text, size=11, bold=True, color="FFFFFF")
            ws['A1'].fill = PatternFill(start_color=M_BLUE_DARK, end_color=M_BLUE_DARK, fill_type="solid")
            ws['A1'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            ws.row_dimensions[1].height = 25
            ws.row_dimensions[2].height = 25

            dv = DataValidation(type="list", formula1='"✅ 是,❌ 否"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f'D4:D{max_row}')
            ws.auto_filter.ref = f"A3:{get_column_letter(ws.max_column)}{max_row}"

            column_widths = {}

            for row in ws.iter_rows(min_row=3, max_row=max_row):
                r_idx = row[0].row
                ws.row_dimensions[r_idx].height = 22

                for cell in row:
                    cell.border = Border(left=Side(style='thin', color=M_BORDER), right=Side(style='thin', color=M_BORDER),
                                         top=Side(style='thin', color=M_BORDER), bottom=Side(style='thin', color=M_BORDER))

                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=False)

                    val = str(cell.value) if cell.value is not None else ""

                    current_width = sum(2.3 if re.search(r'[\u4e00-\u9fff]', c) else 1.3 for c in val)

                    if r_idx == 3:
                        current_width += 8

                    column_widths[cell.column] = max(column_widths.get(cell.column, 0), current_width)

                    if r_idx == 3:
                        cell.fill = PatternFill(start_color=M_BLUE_DARK, end_color=M_BLUE_DARK, fill_type="solid")
                        cell.font = Font(name='Segoe UI', bold=True, color="FFFFFF", size=12)
                    else:
                        is_missing = ("❌" in str(row[3].value))
                        if cell.column <= 3: cell.fill = PatternFill(start_color=M_BLUE_LIGHT, end_color=M_BLUE_LIGHT, fill_type="solid")
                        elif cell.column == 4: cell.fill = PatternFill(start_color=M_PINK, end_color=M_PINK, fill_type="solid")
                        else: cell.fill = PatternFill(start_color=M_GREEN, end_color=M_GREEN, fill_type="solid")

                        f_color = "B55D5D" if is_missing else M_TEXT
                        f_name = '微軟正黑體' if re.search(r'[\u4e00-\u9fff]', val) else 'Segoe UI'
                        cell.font = Font(name=f_name, size=11, color=f_color)

            for col_idx, final_w in column_widths.items():
                ws.column_dimensions[get_column_letter(col_idx)].width = final_w

            # 視覺化分頁
            ws2 = writer.book.create_sheet(title="ResultVisual")
            ws2.merge_cells('A1:M3')
            visual_header = "【電路圖目視檢驗表單】\n 🔹 亮黃色標記：已存在於 BOM 清單中。 🔹 無標記：圖中存在但 BOM 遺漏之元件。"
            apply_mixed_font(ws2['A1'], visual_header, size=11, bold=True)
            ws2['A1'].fill = PatternFill(start_color=M_BEIGE, end_color=M_BEIGE, fill_type="solid")
            ws2['A1'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            ws2.row_dimensions[1].height = 45
            ws2.freeze_panes = 'A4'

            current_row = 5
            for img_path in page_images:
                img = Image(img_path)

                target_display_width = 1000  # Excel 顯示寬度
                aspect_ratio = img.height / img.width
                img.width = target_display_width
                img.height = int(target_display_width * aspect_ratio)

                ws2.add_image(img, f'B{current_row}')
                row_span = int(img.height / 15)
                current_row += row_span + 2

        doc.close()
        for p in page_images:
            if os.path.exists(p): os.remove(p)
        print(f"✨ 任務完成！報表：{OUTPUT_EXCEL}")

    except Exception as e:
        print(f"❌ 錯誤: {e}")

if __name__ == "__main__":
    start_process()

from google.colab import files
files.download(OUTPUT_EXCEL)

🚀 啟動完整校正程序：分析 SCH.pdf ...
✨ 任務完成！報表：SCH & BOM 比對結果.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>